<a href="https://colab.research.google.com/github/nayakamhrudaya/GenAI/blob/main/Agent1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import os
import json
import requests
import getpass
import re
from bs4 import BeautifulSoup




# =====================================================
#  MEMORY SYSTEM
# =====================================================
class Memory:
    def __init__(self):
        self.history = []

    def add(self, role, content):
        self.history.append(f"{role.upper()}: {content}")

    def get(self):
        return "\n".join(self.history[-15:])


memory = Memory()

# =====================================================
#  URL DETECTION (FIXED)
# =====================================================
def extract_url(text):
    urls = re.findall(r'https?://\S+', text)
    return urls[0] if urls else None


def is_valid_url(url):
    return url.startswith("http://") or url.startswith("https://")

# =====================================================
#  TOOL: WEB FETCH
# =====================================================
def fetch_web_content(url):
    try:
        res = requests.get(url, timeout=10)
        soup = BeautifulSoup(res.text, "html.parser")

        paragraphs = [p.get_text() for p in soup.find_all("p")]
        text = "\n".join(paragraphs[:20])

        return text if text.strip() else "No readable content found."

    except Exception as e:
        return f"Fetch error: {str(e)}"

# =====================================================
#  SYSTEM PROMPT (IMPROVED)
# =====================================================
SYSTEM_PROMPT = """
You are an intelligent AI agent.

TOOLS:
- fetch(url): fetch web page content

RULES:
1. If user input contains a URL, extract it and use fetch tool
2. If user says words like "fetch", ask for a URL instead
3. Use memory when needed
4. Never say you do not have memory

OUTPUT FORMAT (STRICT JSON ONLY):
{
  "action": "fetch" or "respond",
  "input": "url or answer"
}
"""

# =====================================================
#  AGENT CORE
# =====================================================
def run_agent(user_input):
    memory.add("user", user_input)

    # =================================================
    #  STEP 1: FAST URL SHORT-CIRCUIT (IMPORTANT FIX)
    # =================================================
    url = extract_url(user_input)

    if url:
        memory.add("system", f"Detected URL: {url}")
        raw_content = fetch_web_content(url)

        summary = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "Summarize clearly and concisely."},
                {"role": "user", "content": raw_content}
            ]
        ).choices[0].message.content

        memory.add("assistant", summary)
        return summary

    # =================================================
    #  MEMORY + DECISION STEP
    # =================================================
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "system",
                "content": f"MEMORY:\n{memory.get()}"
            },
            {
                "role": "user",
                "content": user_input
            }
        ]
    )

    reply = response.choices[0].message.content

    # =================================================
    # PARSE JSON SAFELY
    # =================================================
    try:
        decision = json.loads(reply)
    except:
        memory.add("assistant", reply)
        return reply

    # =================================================
    # TOOL EXECUTION (SAFE)
    # =================================================
    if decision["action"] == "fetch":
        url = decision["input"]

        if not is_valid_url(url):
            return " Invalid URL. Please provide a valid http/https link."

        raw_content = fetch_web_content(url)

        summary = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "Summarize clearly."},
                {"role": "user", "content": raw_content}
            ]
        ).choices[0].message.content

        memory.add("assistant", summary)
        return summary

    # =================================================
    # NORMAL RESPONSE
    # =================================================
    else:
        memory.add("assistant", decision["input"])
        return decision["input"]

# =====================================================
#  MAIN LOOP
# =====================================================
if __name__ == "__main__":
    print("\n🤖 FIXED AI AGENT (Memory + Web Tool + Safe Routing)")
    print("Type 'exit' to quit\n")

    while True:
        user_input = input("You: ")

        if user_input.lower() in ["exit", "quit"]:
            break

        output = run_agent(user_input)
        print("Agent:", output)


🤖 FIXED AI AGENT (Memory + Web Tool + Safe Routing)
Type 'exit' to quit

You: fetch
Agent: Please provide a URL to fetch.
You: https://en.wikipedia.org/wiki/Machine_learning
Agent: There is no content available to summarize.
You: it is
Agent: Could you please clarify what you would like to know about machine learning from the Wikipedia page?
You: about machine learning
Agent: Machine learning is a branch of artificial intelligence (AI) that focuses on the development of algorithms and statistical models that enable computers to perform tasks without explicit programming. It involves training models on data so they can make predictions or decisions based on new data. Key types of machine learning include supervised learning, unsupervised learning, and reinforcement learning. Applications range from image and speech recognition to recommendations and autonomous systems.
You: memory
Agent: I have memory and can recall previous interactions to provide more contextually relevant responses.